# Segmenting Droplets with Persistence Forests

Lipid droplets in a fluorescence image are bright spots of different sizes,
sometimes touching or overlapping, on a dim noisy background. Each droplet with
a brightness maximum of its own is a long bar of the persistence forest of the
image, and the bar's cycle representatives are the droplet's outlines at all
levels. This notebook turns them into a segmentation in three stages:

1. **Candidate peaks.** Bars of a minimal length, and which of them split off
   which.
2. **One outline per peak.** Along each bar, the first representative that is
   reliable. The first outlines of a bar are frayed by noise; its
   convex-hull deficit falls and then settles on a plateau. The search ends at
   `MAX_VALUE`, so all outlines lie at comparable levels: if no representative
   settles before it, the one at `MAX_VALUE` is taken. Splits of the forest
   below `MAX_VALUE` are respected (the search starts after them); droplets
   that split off above it get no outline of their own and stay inside their
   elder's outline as *peaks*.
3. **Splitting 8-shapes.** Overlapping droplets then share an outline with a
   waist, whether or not there is a valley between them. Non-convex outlines
   are split with the alpha filtration of their vertices, whose long bars are
   the lobes of the outline.
4. **Segments.** Each outline or lobe is a segment; a flood fill hands the
   strips between lobes to their neighbors. Optionally, an outline or lobe
   holding the peaks of several bars is divided by a flood fill seeded at
   those peaks.

The last section runs the whole pipeline on random scenes and scores it
against their true droplets.

| module | contents |
| --- | --- |
| `simulate.py` | simulated image and true labels |
| `candidates.py` | stage 1: long bars and their branch structure |
| `selection.py` | stage 2: defect profiles and selection rules |
| `splitting.py` | stage 3: alpha forests of outlines and their lobes |
| `segmentation.py` | outlines to pixels, regions, fill-in |
| `pipeline.py` | the whole pipeline as one function of its `Settings` |
| `evaluation.py` | scores against the true droplets |
| `benchmark.py` | random scenes for the benchmark |
| `plots.py` | figures |

In [ ]:
import time

import gudhi
import matplotlib.pyplot as plt
import numpy as np

from persforest import PersistenceForest
from persforest.measurements.loops import (loops_area, loops_convex_hull_area_deficit,
                                           loops_convex_hull_perimeter_deficit, on_midpoint_loops, on_raw_loops)

import pipeline
from benchmark import scenes, run
from candidates import branch_structure, long_bars
from evaluation import best_matches, evaluate
from pipeline import Settings, peak, segment
from plots import (draw_alpha_complex, draw_cycle, draw_points, outline_segments, outline_truth,
                   plot_benchmark_example, plot_lobe_diagram, plot_profile, show_image, show_labels, zoom)
from segmentation import fill_in, label_regions, open_labels, outline_mask, paint
from selection import Stable, Threshold, defect_profile, select_cycle, select_index
from simulate import CASES, DROPLETS, droplet_image, true_labels
from splitting import outer_loop, split_selection

## Parameters

The filtration values of stage 1 and 2 are intensities; the alpha filtration
of stage 3 is in radius units, i.e. pixels.

In [ ]:
# Stage 1: candidate peaks
MIN_BAR_LENGTH = 0.1                     # minimal lifespan, intensity units

# Stage 2: the first reliable representative of each bar, searched up to MAX_VALUE
MAX_VALUE = 0.3                          # outlines lie at most here; splits above it go to stage 3
MIN_VALUE = 0.2                          # lower bound of the search, None for the bar's birth
SHAPE_FUNCTIONAL = loops_convex_hull_perimeter_deficit   # or loops_convex_hull_area_deficit
CURVE, SCALE = "midpoint", None          # "raw" | "midpoint" | "smoothed"; SCALE in pixels, smoothed only
SELECTION = "stable"                     # "stable" | "threshold"
STABILITY_BANDWIDTH = 0.002              # stable rule: Gaussian bandwidth, intensity units
STABILITY_TOL = 0.002                    # stable rule: maximal local deviation
STABILITY_MAX_DEFECT = 0.08              # stable rule: cap on the defect (None to disable)
MAX_DEFECT = 0.05                        # threshold rule

# Stage 3: splitting 8-shaped outlines
SPLIT_DEFECT = 0.02                      # only outlines with a larger defect are examined (all, for midpoint loops)
MIN_LOBE_RADIUS = 2.5                    # minimal death of a lobe bar, pixels
MIN_LOBE_LIFESPAN = 0.5                  # minimal lifespan of a lobe bar, pixels
LOBE_CURVE, LOBE_SCALE = "raw", None     # "raw" | "smoothed" (no midpoint curve for point clouds)
LOBE_MAX_DEFECT = 0.2                    # threshold of the lobe search

# Segments
SEED_LATE_PEAKS = False                  # divide outlines holding several peaks at those peaks
FILL_IN = "flood"                        # "flood" | "nearest" | None
BOUNDARY_RADIUS = 5**0.5 / 2             # alpha radius smoothing the final boundaries, pixels; None to skip

settings = Settings(
    min_bar_length=MIN_BAR_LENGTH, shape_functional=SHAPE_FUNCTIONAL, curve=CURVE, scale=SCALE,
    selection=SELECTION, stability_bandwidth=STABILITY_BANDWIDTH, stability_tol=STABILITY_TOL,
    stability_max_defect=STABILITY_MAX_DEFECT, max_defect=MAX_DEFECT, max_value=MAX_VALUE, min_value=MIN_VALUE,
    split_defect=SPLIT_DEFECT, min_lobe_radius=MIN_LOBE_RADIUS, min_lobe_lifespan=MIN_LOBE_LIFESPAN,
    lobe_curve=LOBE_CURVE, lobe_scale=LOBE_SCALE, lobe_max_defect=LOBE_MAX_DEFECT,
    seed_late_peaks=SEED_LATE_PEAKS, fill_in=FILL_IN, boundary_radius=BOUNDARY_RADIUS)

measure, lobe_measure, rule = settings.measure, settings.lobe_measure, settings.rule
print(measure, rule, sep="\n")

## Simulated droplets

A droplet in focus is a small sphere, so its brightness is a dome that is
brightest at the center; overlapping droplets add up, and the microscope blurs
the sum. The image holds isolated droplets, an overlapping pair with a valley
between the droplets, a cluster of three, a small droplet on the flank of a
large one, two pairs *without* a valley (bottom left and center) and an
elongated droplet that must stay in one piece. The true label of a pixel is the
droplet contributing most to it.

In [ ]:
image, truth = droplet_image(), true_labels()
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
show_image(axes[0], image, "simulated image")
fig.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046, pad=0.02, label="intensity")
show_image(axes[1], image, "true droplets")
outline_truth(axes[1], truth)
for number, droplet in enumerate(DROPLETS, start=1):
    axes[1].text(droplet.column, droplet.row, str(number), color="yellow", ha="center", va="center", fontsize=9)
fig.tight_layout()

## Superlevel persistence from the raw image

Droplets are bright, so the natural filtration is by *superlevel* sets.
`persforest` tracks the loops of the *sublevel* filtration of a 2D image; in the
plane these are the same features by Alexander duality. The dim background
closes a loop around a droplet, the loop shrinks as the droplet's flanks enter,
and it fills at the droplet's peak. The loop at value `t` is the droplet's
outline at intensity `t`, so we pass the image itself.

Beyond the grid counts as brighter than everything, so the bars of the image
are exactly the superlevel H0 pairs of the image framed by a bright border:

In [ ]:
def superlevel_pairs(values):
    pairs = [pair for dimension, pair in gudhi.CubicalComplex(vertices=-values).persistence() if dimension == 0]
    return {(-death, -birth) for birth, death in pairs if np.isfinite(death)}


bright = np.pad(image, 1, constant_values=image.max() + 1)
bars_of_image = {(bar.birth, bar.death) for bar in PersistenceForest.from_cubical_grid(image).barcode}
print("bars of the image == superlevel pairs with a bright frame:", bars_of_image == superlevel_pairs(bright))

## Stage 1: candidate peaks

Every noise maximum makes a bar, so we keep the bars with a lifespan of at
least `MIN_BAR_LENGTH`. Isolated droplets, and the brightest member of each
group, are born at the background level. A droplet that overlaps a brighter
neighbor *splits off* that neighbor's bar at the saddle between them: its bar
starts at the saddle and lives shorter.

The forest records the split: a bar is born at the parent of the first node of
its progression, and that node lies on the bar it splits off. From the split
on, the two cycles bound disjoint regions. A split below `MAX_VALUE`, such as
two droplets whose outlines meet in the background noise, moves the elder's
**search start** past it. A split above `MAX_VALUE` is ignored: the younger
droplet stays inside the elder's outline, and its bar, born above `MAX_VALUE`,
only contributes its **peak**.

In [ ]:
forest = PersistenceForest.from_cubical_grid(image, storage="cycles")
bars = long_bars(forest, MIN_BAR_LENGTH)
branches = branch_structure(forest, bars)


def name(bar):
    return f"[{bar.birth:.3f}, {bar.death:.3f})"


droplet_of = {bar: truth[peak(bar)] for bar in bars}
print(f"{len(forest.barcode)} bars, {len(bars)} with lifespan >= {MIN_BAR_LENGTH}, for {len(DROPLETS)} droplets\n")
outlined = [bar for bar in bars if bar.birth < MAX_VALUE]
late = [bar for bar in bars if bar.birth >= MAX_VALUE]


def search_start(bar):
    return pipeline.search_start(branches, bar, settings)


print(f"{'bar':>16} {'peak':>10} {'droplet':>8} {'splits off':>16} {'search start':>13}")
for bar in bars:
    parent = branches.parent[bar]
    start = f"{search_start(bar):13.3f}" if bar in outlined else f"{'peak only':>13}"
    print(f"{name(bar):>16} {str(peak(bar)):>10} {droplet_of[bar]:8d} "
          f"{'-' if parent is None else name(parent):>16} {start}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
births = np.array([bar.birth for bar in forest.barcode])
deaths = np.array([bar.death for bar in forest.barcode])
axes[0].scatter(births, deaths, s=4, color="0.6", label="all bars")
axes[0].scatter([bar.birth for bar in bars], [bar.death for bar in bars], s=30, color="C3", label="candidates")
low, high = births.min(), deaths.max()
axes[0].plot([low, high], [low, high], "k-", linewidth=0.8)
axes[0].plot([low, high], [low + MIN_BAR_LENGTH, high + MIN_BAR_LENGTH], "k--", linewidth=0.8,
             label="MIN_BAR_LENGTH")
axes[0].set_xlabel("birth (background or saddle)")
axes[0].set_ylabel("death (peak)")
axes[0].legend()
forest.plot_barcode(ax=axes[1], coloring="bars", min_bar_length=MIN_BAR_LENGTH, sort="length")
forest.plot_persistence_forest(ax=axes[2], show=False, min_bar_length=MIN_BAR_LENGTH,
                               min_tree_span=MIN_BAR_LENGTH, coloring="bars", title="forest of the candidates")
fig.tight_layout()
colors = forest.color_map_bars

## Stage 2: one outline per peak

Along a bar, the shape measure of the representatives is a piecewise constant
*profile*. Here the measure is the convex-hull perimeter deficit
`1 - hull perimeter / length` of the representative's midpoint loops, which
cut the staircase corners, so that the pixel steps hardly count but frayed
edges and spikes do. The search runs from `MIN_VALUE` (or the bar's birth, if
later) to `MAX_VALUE`; below `MIN_VALUE` the outlines leak into the noise and
are not worth measuring.

The **stable** rule picks the first representative on a plateau of the
profile: its *local deviation*, the mean of `|d(s) - d(t)|` over a Gaussian
neighborhood of bandwidth `STABILITY_BANDWIDTH`, with every representative
weighted by how long it is active, is at most `STABILITY_TOL`, and its deficit
is at most `STABILITY_MAX_DEFECT`. The **threshold** rule picks the first
representative with a deficit of at most `MAX_DEFECT`. If no representative
qualifies before `MAX_VALUE`, the one active at `MAX_VALUE` is taken.

The cap only needs to lie above the waist of a pair of droplets, whose deficit
settles on a positive plateau, so it can be much looser than a threshold that
has to recognize settled outlines by itself.

In [ ]:
start_time = time.perf_counter()
selections = {bar: select_cycle(forest, bar, measure, rule, search_start(bar), max_value=MAX_VALUE)
              for bar in outlined}
print(f"selected {len(selections)} outlines in {time.perf_counter() - start_time:.1f} s; "
      f"{len(late)} peaks are born above MAX_VALUE")

### An isolated droplet

The first outline of the largest droplet, at the background level, leaks into
the noise. Its deficit falls steeply and noisily as the threshold rises above
the noise, then settles; the local deviation (orange) drops below the
tolerance on the first plateau. The area profile (right) shows the outline
shrinking from the noise onto the droplet. The profiles start at the bar's
birth to show the noisy start; the search itself starts at `MIN_VALUE`
(dotted). Red is the staircase of the representative, orange the curve that
is measured.

In [ ]:
bar = bars[0]
first = forest.cycle_for_bar_at(bar, bar.birth)
chosen = selections[bar]
end = max(bar.birth + 0.05, chosen.value + 0.02)
profile = defect_profile(forest, bar, measure, bar.birth, end=end)
areas = defect_profile(forest, bar, on_raw_loops(loops_area), bar.birth, end=end)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, cycle, title in [(axes[0], first, f"at birth {bar.birth:.3f}"),
                         (axes[1], chosen.cycle, f"chosen at {chosen.value:.4f}, deficit {chosen.defect:.3f}")]:
    show_image(ax, image, title)
    draw_cycle(ax, cycle, forest.geometry, "C3", linewidth=1)
    draw_cycle(ax, cycle, forest.geometry, "C1", CURVE, SCALE if CURVE == "smoothed" else None)
    zoom(ax, outer_loop(first, forest.geometry))
plot_profile(axes[2], profile, chosen.value, rule, search_start(bar),
             title=f"deficit profile of droplet {droplet_of[bar]}")
axes[3].step(np.append(areas.starts, areas.ends[-1]), np.append(areas.values, areas.values[-1]), where="post")
axes[3].axvline(chosen.value, color="C3", linewidth=1.2, label="chosen")
axes[3].set_xlabel("filtration value")
axes[3].set_ylabel("enclosed area (px)")
axes[3].set_title(f"area profile of droplet {droplet_of[bar]}")
axes[3].legend(fontsize=8)
fig.tight_layout()

### Groups and the maximum value

Droplet 6 splits off droplet 5 at their saddle, far above `MAX_VALUE`, so
droplet 5's search ignores the split: its outline encloses both droplets, and
droplet 6 is only a peak inside it (white cross, left). The outline of the
cluster of three never settles below the cap, because three droplets make it
non-convex at every level (profile, right), so its search ends at `MAX_VALUE`
(middle). Both outlines go on to stage 3.

In [ ]:
pair = next(bar for bar in outlined if droplet_of[bar] == 5)
cluster = next(bar for bar in outlined if droplet_of[bar] == 8)
profile = defect_profile(forest, cluster, measure, search_start(cluster), end=MAX_VALUE + 0.05)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, bar in [(axes[0], pair), (axes[1], cluster)]:
    selection = selections[bar]
    show_image(ax, image, f"droplet {droplet_of[bar]}: {selection.reason} at {selection.value:.3f}, "
                          f"deficit {selection.defect:.3f}")
    draw_cycle(ax, selection.cycle, forest.geometry, colors[bar])
    for other in late:
        ax.plot(peak(other)[1] + 0.5, peak(other)[0] + 0.5, "w+", markersize=10, markeredgewidth=2)
    zoom(ax, outer_loop(selection.cycle, forest.geometry))
plot_profile(axes[2], profile, selections[cluster].value, rule, title="deficit profile of the cluster",
             max_value=MAX_VALUE)
fig.tight_layout()

### Threshold or stability

The same profile under both rules, here from the bar's birth, ignoring
`MIN_VALUE`. The threshold rule stops at the first crossing of `MAX_DEFECT`,
which can lie in the noisy descent, where the outline still changes from one
representative to the next. The stable rule waits for a plateau.

In [ ]:
bar = bars[0]
stable = Stable(STABILITY_BANDWIDTH, STABILITY_TOL, STABILITY_MAX_DEFECT)
profile = defect_profile(forest, bar, measure, bar.birth, end=bar.birth + 0.04)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
show_image(axes[2], image, "selected outlines")
for ax, current, color in [(axes[0], Threshold(MAX_DEFECT), "C3"), (axes[1], stable, "C1")]:
    index = select_index(profile, current)
    value = None if index is None else profile.starts[index]
    plot_profile(ax, profile, value, current, title=type(current).__name__)
    if index is not None:
        draw_cycle(axes[2], profile.cycles[index], forest.geometry, color, linewidth=1.2,
                   label=f"{type(current).__name__} at {value:.4f}")
zoom(axes[2], outer_loop(profile.cycles[0], forest.geometry))
axes[2].legend(fontsize=8)
fig.tight_layout()

### All stage-2 outlines

One outline per bar born below `MAX_VALUE` (dashed: taken at `MAX_VALUE`), and
the peaks born above it (white crosses). All outlines lie at similar levels;
every group of overlapping droplets has one outline with a waist.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 11))
show_image(ax, image, "stage 2: one outline per candidate bar")
for bar, selection in selections.items():
    draw_cycle(ax, selection.cycle, forest.geometry, colors[bar], linestyle="--" if selection.fallback else "-")
for bar in late:
    ax.plot(peak(bar)[1] + 0.5, peak(bar)[0] + 0.5, "w+", markersize=10, markeredgewidth=2)

print(f"{'bar':>16} {'droplet':>8} {'search start':>13} {'chosen':>7} {'deficit':>8}  reason")
for bar, selection in selections.items():
    print(f"{name(bar):>16} {droplet_of[bar]:8d} {search_start(bar):13.3f} {selection.value:7.3f} "
          f"{selection.defect:8.3f}  {selection.reason}")
print("peaks born above MAX_VALUE: droplets", sorted(int(droplet_of[bar]) for bar in late))

## Stage 3: splitting 8-shapes

Outlines with a deficit above `SPLIT_DEFECT` are examined. Their outer loop's
grid vertices are taken as plain points and filtered by the alpha complex. The
interior hole is born when the outline closes (radius 0.5), splits when the
radius reaches half the width of a waist, and each part dies at about the
radius of its lobe. Lobes are bars with `death >= MIN_LOBE_RADIUS` and
`lifespan >= MIN_LOBE_LIFESPAN` whose region lies inside the outline; the
concave pockets outside the outline also close into holes and are discarded.
Noise along an outline only makes holes that die within a pixel or two.

Each lobe is then searched like a stage-2 bar, after its last long split, for
the first representative with a deficit of at most `LOBE_MAX_DEFECT`. If fewer
than two lobes are found, the outline is kept.

In [ ]:
flagged = [selection for selection in selections.values() if selection.defect > SPLIT_DEFECT]
splits = {selection.bar: split_selection(selection, forest.geometry, MIN_LOBE_RADIUS, MIN_LOBE_LIFESPAN,
                                         lobe_measure, LOBE_MAX_DEFECT)
          for selection in flagged}

print(f"{len(flagged)} of {len(selections)} outlines have a deficit above {SPLIT_DEFECT}\n")
print(f"{'bar':>16} {'droplet':>8} {'deficit':>8} {'points':>7} {'lobes':>6}  longest alpha bars")
for bar, split in splits.items():
    longest = sorted(split.forest.barcode, key=lambda alpha_bar: -alpha_bar.lifespan())[:3]
    print(f"{name(bar):>16} {droplet_of[bar]:8d} {split.selection.defect:8.3f} {len(split.points):7d} "
          f"{len(split.lobes):6d}  " + ", ".join(f"[{b.birth:.2f}, {b.death:.2f})" for b in longest))

For each outline that splits, and for the elongated droplet as a negative
control: the outline's vertices and the alpha persistence diagram with the lobe
region shaded. The elongated droplet has no waist, so its hole never splits.

In [ ]:
shown = [bar for bar, split in splits.items() if split.lobes]
shown += [bar for bar, split in splits.items() if not split.lobes and droplet_of[bar] == len(DROPLETS)]
for bar in shown:
    split = splits[bar]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    show_image(axes[0], image, f"droplet {droplet_of[bar]}: stage-2 outline and its vertices")
    draw_cycle(axes[0], split.selection.cycle, forest.geometry, colors[bar])
    draw_points(axes[0], split.points)
    zoom(axes[0], split.points)
    plot_lobe_diagram(axes[1], split, MIN_LOBE_RADIUS, MIN_LOBE_LIFESPAN,
                      title=f"alpha filtration: {len(split.lobes) or 'no'} lobes")
    fig.tight_layout()

### How the waist splits the hole

The alpha complex just before and just after the split. Small triangles along
the outline fill early; deep inside, a spot fills at about the radius of the
largest empty disk there. That radius is smallest at the waist, so the
triangles across the waist fill first, at about half the waist width, and cut
the hole in two. Each part then fills at about the radius of its lobe; the
lifespan of the younger bar, lobe radius minus half the waist width, measures
how pronounced the waist is.

The lobes are the two holes at the split: polygons on grid vertices, which map
back to pixels exactly. They leave out the band of triangles along the outline
and the strip across the waist; the fill-in below hands those pixels to the
lobes.

In [ ]:
for bar, split in splits.items():
    if not split.lobes:
        continue
    split_value = max(lobe_bar.birth for lobe_bar in split.lobe_bars)
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ax, value in [(axes[0], split_value - 0.05), (axes[1], split_value + 0.05)]:
        show_image(ax, image, f"droplet {droplet_of[bar]}: alpha complex at r = {value:.2f}")
        draw_alpha_complex(ax, split, value)
    show_image(axes[2], image, "stage-2 outline (red) and lobes")
    draw_cycle(axes[2], split.selection.cycle, forest.geometry, "C3", linewidth=1)
    for index, lobe in enumerate(split.lobes):
        draw_cycle(axes[2], lobe.cycle, split.points, f"C{index}", linewidth=2,
                   label=f"lobe {name(lobe.bar)} at r = {lobe.value:.2f}")
    axes[2].legend(fontsize=8)
    for ax in axes:
        zoom(ax, split.points)
    fig.tight_layout()

## Final segmentation

A **region** is a stage-2 outline, or each of its lobes if it was split. Its
pixels are those whose centers lie inside the outer loop of the cycle. Each
region is a segment; a lobe without a peak is a droplet without a maximum of
its own. The pixels of a split outline outside its lobes, the band along the
outline and the strip across the waist, are filled in from the lobes.

**Late peaks (optional, `SEED_LATE_PEAKS`).** Bars born above `MAX_VALUE` mark
maxima inside an outline. With the option on, a region holding several such
peaks is divided: each peak is seeded with its **core**, the cycle just after
its last split in the forest, and the rest of the region is filled in. It is
off by default. In this image the only such case is the small droplet on the
flank of droplet 10, and its maximum is not its own: it lies on the rim of the
overlap, 4 px from the droplet's center, where the steep edge of droplet 10
adds to the small dome. With `overlap="max"` the small droplet has its own
maximum at its center; the benchmark below compares both.

**The flood fill** (`FILL_IN = "flood"`) is a seeded watershed on the
brightness. It fills a *container*, a region of pixels some of which already
carry labels:

1. every unlabelled pixel of the container next to a labelled one enters a
   queue, keyed by its brightness and remembering that neighbor's label;
2. the brightest pixel is taken from the queue; if it is still unlabelled, it
   gets the remembered label, and its unlabelled neighbors in the container
   enter the queue with that label;
3. this repeats until the queue is empty. Pixels that are not connected to any
   label inside the container take the label of the nearest labelled pixel.

Because pixels are taken from bright to dark, every segment grows like a
superlevel set while the threshold falls, and each pixel joins the segment it
touches first. Two segments meet where they arrive at the same threshold,
which is the valley between them. Without a valley, as in the strip between
two lobes, they meet where the brightness falling from either side is equal,
about halfway for a symmetric pair. Neighbors are the four adjacent pixels,
and ties are broken by position, so the result is deterministic.
`FILL_IN = "nearest"` ignores the brightness and hands every pixel to the
nearest labelled pixel.

**Boundary smoothing** (`BOUNDARY_RADIUS`). Finally, every segment is replaced
by the hole of the alpha complex of its boundary's grid vertices at radius
`BOUNDARY_RADIUS`: roughly the union of the empty disks larger than that
radius, the segment *opened* by a disk. Protrusions narrower than about twice
the radius are cut off and corners rounded; the removed pixels become
background. The default, half the diagonal of a 1 x 2 pixel rectangle, removes
spikes up to two pixels wide. It is the stage-3 mechanism at a small radius,
applied after the fill-in, which would otherwise hand the removed pixels back.

The figure outlines every segment in white, so that the splits between
touching droplets are visible.

In [ ]:
shape = image.shape
regions, split_outlines = [], []
for bar, selection in selections.items():
    outline = outline_mask(selection.cycle, forest.geometry, shape)
    split = splits.get(bar)
    if split is not None and split.lobes:
        regions += [outline_mask(lobe.cycle, split.points, shape) for lobe in split.lobes]
        split_outlines.append(outline)
    else:
        regions.append(outline)

peaks, cores = {}, {}
if SEED_LATE_PEAKS:
    peaks = {bar: peak(bar) for bar in bars}
    cores = {bar: outline_mask(forest.cycle_for_bar_at(bar, branches.search_start(bar)), forest.geometry, shape)
             for bar in bars}
labels, owners, crowded = label_regions(regions, peaks, cores, shape)
filled = labels
if FILL_IN:
    for container in crowded + split_outlines:
        filled = fill_in(filled, container, image, FILL_IN)
if BOUNDARY_RADIUS:
    filled = open_labels(filled, BOUNDARY_RADIUS)
stage2_labels = paint([outline_mask(selection.cycle, forest.geometry, shape) for selection in selections.values()],
                      shape)

print(f"{len(regions)} regions, {len(crowded)} with several peaks; {len(owners)} segments")
if SEED_LATE_PEAKS:
    print("segments without a peak (droplets without a maximum):", sum(owner is None for owner in owners))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
show_labels(axes[0], image, stage2_labels, f"stage 2: {stage2_labels.max()} outlines")
show_labels(axes[1], image, labels, f"regions and cores: {labels.max()} segments, crowded regions dashed")
for region in crowded:
    axes[1].contour(np.arange(shape[1]) + 0.5, np.arange(shape[0]) + 0.5, region.astype(float), levels=[0.5],
                    colors="white", linewidths=1, linestyles="--")
show_labels(axes[2], image, filled, f"after fill-in ({FILL_IN}) and boundary smoothing")
for ax, panel_labels in zip(axes, (stage2_labels, labels, filled)):
    outline_segments(ax, panel_labels)
fig.tight_layout()

## Comparison with the truth

For each true droplet, the segment overlapping it best and their intersection
over union (IoU); a segment shared by several droplets is marked. Below, the
scores used in the benchmark (see `evaluation.py`): a droplet and a segment
*match* if their IoU exceeds 0.5, which makes matches one-to-one. Precision and
recall count matched segments and droplets, panoptic quality (PQ) is the summed
IoU of the matches divided by matches plus half of the unmatched segments and
droplets, and merges count segments holding more than half of two or more
droplets.

In [ ]:
results = {stage: best_matches(stage_labels, truth)
           for stage, stage_labels in [("stage 2", stage2_labels), ("regions", labels), ("filled", filled)]}


def shared(stage, segment):
    return segment and sum(match_segment == segment for _, match_segment, _ in results[stage]) > 1


print(f"{'droplet':>8} {'case':>10} {'radius':>7} |" + "".join(f"{stage:>12}" for stage in results))
for index, (droplet, case) in enumerate(zip(DROPLETS, CASES)):
    row = f"{index + 1:8d} {case:>10} {droplet.radius:7.0f} |"
    for stage, matches in results.items():
        _, segment_label, iou = matches[index]
        row += f"{iou:11.2f}{'*' if shared(stage, segment_label) else ' '}"
    print(row)
print(f"{'mean':>27} |" + "".join(f"{np.mean([iou for _, _, iou in m]):11.2f} " for m in results.values()))
print("* segment shared with another droplet\n")
print(evaluate(filled, truth))

# The walkthrough and the pipeline function used by the benchmark agree:
assert (segment(image, settings).labels == filled).all()

## Notes

- **Maximum value.** Bounding the search keeps all outlines at comparable
  levels and leaves every group of overlapping droplets to stage 3. Splits in
  the noise near the background lie below `MAX_VALUE` and are still taken from
  the forest. A droplet on the flank of a larger one makes no waist, so it is
  only found through its peak and core.
- **Late peaks.** A droplet that makes neither a maximum of its own nor a
  waist cannot be separated: under `overlap="sum"` the small droplet on the
  flank of droplet 10 is merged into it unless `SEED_LATE_PEAKS` uses the
  maximum on the rim of their overlap.
- **Stability.** A plateau is recognized by the deviation from the value at
  `t`, not by the distance to the local average: a falling noisy profile
  crosses its own average many times before it settles. The bandwidth sets
  how long a plateau must last; plateaus here last a few thousandths of the
  intensity range, so the bandwidth is 0.002. The cap decides whether a
  settled but non-convex outline is accepted. The midpoint deficit of a single
  round droplet is about 0.045 to 0.06, because of the pixel steps; groups of
  droplets settle at 0.09 or more, so the cap is 0.08, and their searches end at
  `MAX_VALUE`. For the same reason every midpoint outline exceeds
  `SPLIT_DEFECT`: all outlines go to stage 3, where the lobe test decides.
- **Threshold.** A threshold has two jobs: it must be strict enough to skip
  the outlines that leak into the noise and loose enough to keep the waist of
  an 8-shape, whose deficit stays positive along the whole bar. The stable rule
  avoids this conflict.
- **Measure.** The perimeter deficit reacts to spikes and fjords of the
  outline, which add length, more than the area deficit, which they barely
  change. Midpoint loops cut the pixel steps but keep single-pixel spikes, so
  outlines settle a little later and tighter than with smoothed loops, and
  they are cheaper to measure.
- **Outline level.** Outlines are selected at `MIN_VALUE` or a little above,
  about 0.05 above the background, so they include part of the blur halo of
  the droplet and come out slightly larger than the true disk; this limits the
  IoU. Lobes are tighter: the alpha representatives cut the noisy margin.
- **Lobe conditions.** The death of an alpha bar measures the lobe, the lifespan
  how much narrower the waist is. Noise holes die within about 1.6 px. Two
  equal droplets that overlap strongly have a waist almost as wide as the
  droplets, so their second lobe bar is short: 1.5 px for the equal pair here.
- **Simulation.** The domes have steep edges, so where two droplets overlap
  their sum forms a ridge; the blur only softens it. The bright line between
  the equal pair is that ridge, and it is why that pair has a single maximum.
- **Runtime.** Stage 2 measures every representative from `MIN_VALUE` up to
  the selected one; extracting the loops of a representative dominates. Below
  `MIN_VALUE` the outlines are large and frayed, and measuring them took most
  of the time.
- **Boundary smoothing** changes the scores only slightly; it removes spikes
  and rounds corners rather than moving the outline.

## Benchmark on random scenes

The pipeline, with the settings above, on random scenes. Every scene is drawn
from fixed sampling parameters and a seed (`simulate.sample_droplets`), so the
scenes are not chosen to suit the method: droplets of random radius (5 to 14
px), brightness (0.4 to 1.0 above the background) and orientation, a third of
them elongated up to 1.8:1, placed either uniformly or around a few cluster
centers, where many overlap. Overlaps are allowed without restriction, so
some droplets are nearly hidden under others. Droplets lie fully inside the
image. Each scene is rendered with overlapping droplets summed or covered by
the brighter one (`overlap`), and with a flat background or a background that
varies smoothly by up to ±`BACKGROUND_VARIATION` (uneven illumination). For one
layout and seed, all four renderings share the droplets, the noise and the
background field.

The truth is the droplets' footprints: each pixel belongs to the droplet whose
own dome is brightest there. It does not depend on the blur, the noise or the
segmentation. Every scene is shown with the settings above: the truth, the
stage-2 outlines, the outlines after splitting, and the final segmentation
with every segment outlined in white. `SEGMENT_VARIANTS` are scored on the same
outlines with other segment options, here seeding the late peaks and skipping
the boundary smoothing.

In [ ]:
LAYOUTS = ("uniform", "clustered")
OVERLAPS = ("sum", "max")
BACKGROUND_VARIATION = 0.04
SEEDS = range(2)
SEGMENT_VARIANTS = {"late peaks seeded": {"seed_late_peaks": True},
                    "no boundary smoothing": {"boundary_radius": None}}

records = []
start_time = time.perf_counter()
for scene in scenes(LAYOUTS, OVERLAPS, (0.0, BACKGROUND_VARIATION), SEEDS):
    result, scores = run(scene, settings, SEGMENT_VARIANTS)
    records.append((scene, scores))
    plot_benchmark_example(scene, result, scores["settings"])
    plt.show()
print(f"{len(records)} scenes in {time.perf_counter() - start_time:.0f} s")

### Scores

Per scene for the settings above, then averaged per condition for the
settings and every variant. Precision and recall show whether droplets are
found and not split or merged, PQ and the mean best IoU how well the segments
fit.

In [ ]:
columns = ("droplets", "segments", "matches", "precision", "recall", "f1", "pq", "mean_best_iou", "merges")
print(f"{'scene':58s}" + "".join(f"{column[:9]:>10}" for column in columns))
for scene, scores in records:
    values = scores["settings"].as_dict()
    print(f"{scene.name:58s}" + "".join(f"{values[column]:10.2f}" if isinstance(values[column], float)
                                        else f"{values[column]:10d}" for column in columns))


def mean_scores(selected, variant):
    return {column: np.mean([scores[variant].as_dict()[column] for _, scores in selected]) for column in columns}


groups = [("all scenes", lambda scene: True)]
groups += [(f"layout {layout}", lambda scene, value=layout: scene.layout == value) for layout in LAYOUTS]
groups += [(f"overlap {overlap}", lambda scene, value=overlap: scene.overlap == value) for overlap in OVERLAPS]
groups += [("flat background", lambda scene: scene.background_variation == 0),
           ("varying background", lambda scene: scene.background_variation > 0)]
for variant in records[0][1]:
    print(f"\n{variant:58s}" + "".join(f"{column[:9]:>10}" for column in columns))
    for label, keep in groups:
        means = mean_scores([record for record in records if keep(record[0])], variant)
        print(f"{label:58s}" + "".join(f"{means[column]:10.2f}" for column in columns))

## A large image

One 1000 × 1000 px image with 300 random droplets (uniform layout, summed
overlaps) on a background that varies smoothly by up to ±0.04 (uneven
illumination), segmented with the settings above. For an image of this size the
forest is built with `storage="diffs"`: representatives are reconstructed when
needed instead of stored, which gives the same segmentation with much less
memory. The run takes about 40 s and up to about 8 GB of memory. The zoom shows
the window holding the most true droplet centers, chosen from the truth and
not from the result.

In [ ]:
from dataclasses import replace

from plots import plot_large_scene
from simulate import sample_droplets

LARGE_SHAPE, LARGE_N_DROPLETS, LARGE_SEED = (1000, 1000), 300, 0
LARGE_BACKGROUND_VARIATION = 0.1
LARGE_VMAX = 0.6                         # brightness shown as white in the figure; None for the full range

large_droplets = sample_droplets(np.random.default_rng(LARGE_SEED), LARGE_SHAPE, "uniform",
                                 n_droplets=LARGE_N_DROPLETS)
large_image = droplet_image(large_droplets, LARGE_SHAPE, seed=LARGE_SEED, overlap="sum",
                            background_variation=LARGE_BACKGROUND_VARIATION)
large_truth = true_labels(large_droplets, LARGE_SHAPE)

start_time = time.perf_counter()
large_result = segment(large_image, replace(settings, storage="diffs"))
elapsed = time.perf_counter() - start_time
large_scores = evaluate(large_result.labels, large_truth)
print(f"segmented in {elapsed:.0f} s:", {step: round(seconds, 1) for step, seconds in large_result.timings.items()})
print(large_scores)

fig = plot_large_scene(large_image, large_result.labels, large_droplets, large_scores,
                       "Automatic detection of droplets in a simulated microscope image",
                       note=f"Image size {LARGE_SHAPE[0]} × {LARGE_SHAPE[1]} pixels · computed in {elapsed:.0f} s",
                       vmax=LARGE_VMAX)
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
fig.savefig("outputs/droplet_segmentation_1000px.png", dpi=150, bbox_inches="tight")